# Replicate Published Results

Before trusting what a method returns, rerun the same code on data whose results have been published by others, and compare the numbers one by one.

This tutorial is based on step 2 of the workflow **Linear Regression: a production function** (Compose > Workflows > Search Library). Its code is the Linear Regression panel's **Replicate** Next Step (Next Steps › Compare), on the Clustered or Driscoll-Kraay tab.

In [ ]:
Pollis.packages("Regress" => "https://github.com/gragusa/Regress.jl", "PollisDatasets", "CovarianceMatrices" => "https://github.com/Trumpingtons/CovarianceMatrices.jl"; forks = ("CovarianceMatrices",))

## Why published results

Rewriting the same algorithm or method in another language is not a check: both versions apply the same formulas to the same data, so they agree even when the formulas, a default or the data are wrong. A published table was computed by someone else, with other software, on the original data; matching it checks the data, the estimates and the options together.

The check also reaches beyond this panel: the packages behind it (here Regress.jl and CovarianceMatrices.jl) are tested on the same numbers.

## The model

The Code Example of the **Clustered** tab: a production function for the 48 US states, 1970-1986 (Munnell, 1990), with standard errors clustered by state.

In [ ]:
using Regress, PollisDatasets

data = dataset("produc");  # one row per state and year; datasetinfo("produc") describes the columns
model = Regress.ols(data, @formula(log(gsp) ~ log(pcap) + log(pc) + log(emp) + unemp), save_cluster=:state) + vcov(CR1(:state));  # production function, cluster-robust by state
display(coeftable(model))

## 1. Baltagi (2005, p. 25, Table 2.6)

Baltagi's textbook (Section 2.6.3) reports the pooled OLS estimates of this model on Munnell's data, with classical standard errors, to 3 decimals. The clustered and classical fits share the coefficients; the classical standard errors are computed from the same model by formula.

Each value is shown in full, rounded to the published 3 decimals, and beside the published number.

In [ ]:
using Downloads, LinearAlgebra, Printf

# Baltagi (2005, p. 25, Table 2.6), OLS row: this model on Munnell's data, classical standard errors, published to 3 decimals
baltagi = Dict("log(pcap)" => (0.155, 0.017), "log(pc)" => (0.309, 0.010), "log(emp)" => (0.594, 0.014), "unemp" => (-0.007, 0.001));
X  = modelmatrix(model);
s2 = deviance(model) / dof_residual(model);
classical = sqrt.(s2 .* diag(inv(transpose(X) * X)));  # Baltagi reports classical standard errors
@printf("%-12s%12s%10s%11s%12s%10s%11s\n", "Baltagi", "Coef", "rounded", "published", "SE", "rounded", "published")
for (i, name) in enumerate(coefnames(model))
	haskey(baltagi, name) || continue  # the intercept is not published
	b, se = baltagi[name]
	same = round(coef(model)[i], digits=3) == b && round(classical[i], digits=3) == se
	@printf("%-12s%12.6f%10.3f%11.3f%12.6f%10.3f%11.3f  %s\n", name, coef(model)[i], coef(model)[i], b, classical[i], classical[i], se, same ? "same to 3 decimals" : "DIFFERENT")
end

## 2. Petersen's test data

Petersen publishes on his Kellogg web page, as a companion to Petersen (2009), a test dataset (5000 firm-years, 500 firms over 10 years) with the OLS, White and clustered standard errors in tables headed "OLS Coefficients and Standard Errors", "OLS Coefficients and White Standard Errors" and "OLS Coefficients and Standard Errors Clustered by Firm" (and by Year): the usual benchmark for cluster-robust code. It checks the standard errors of the model above on data that has nothing to do with it.

In [ ]:
using Downloads

# Petersen's test data (Kellogg web page, tables of OLS, White and clustered standard errors): 5000 firm-years, published estimates for x to 4 decimals
file = Downloads.download("https://www.kellogg.northwestern.edu/faculty/petersen/htm/papers/se/test_data.txt");
rows = split.(readlines(file));  # firm, year, x, y
test = (firmid = parse.(Int, getindex.(rows, 1)), year = parse.(Int, getindex.(rows, 2)), x = parse.(Float64, getindex.(rows, 3)), y = parse.(Float64, getindex.(rows, 4)));
ols = Regress.ols(test, @formula(y ~ x));  # Regress reports White (HC1) standard errors by default
k   = findfirst(==("x"), coefnames(ols));
Xt  = modelmatrix(ols);
petersen = ["Coefficient"       => (coef(ols)[k], 1.0348),
	"OLS SE"            => (sqrt(deviance(ols) / dof_residual(ols) * inv(transpose(Xt) * Xt)[k, k]), 0.0286),
	"White SE"          => (stderror(ols + vcov(HC1()))[k], 0.0284),
	"Firm-clustered SE" => (stderror(Regress.ols(test, @formula(y ~ x), save_cluster=:firmid) + vcov(CR1(:firmid)))[k], 0.0506),
	"Year-clustered SE" => (stderror(Regress.ols(test, @formula(y ~ x), save_cluster=:year) + vcov(CR1(:year)))[k], 0.0334)];
@printf("%-20s%12s%10s%11s\n", "Petersen: x", "Julia", "rounded", "published")
for (label, (value, published)) in petersen
	@printf("%-20s%12.6f%10.4f%11.4f  %s\n", label, value, value, published, round(value, digits=4) == published ? "same to 4 decimals" : "DIFFERENT")
end

## Reading the results

- **"same to 3 decimals"** (or 4) means equal after rounding to the precision published, not identical: the full values are in the first column, and nothing beyond the published digits is checked.
- **"DIFFERENT"** calls for an explanation before any result is used. Common causes: another version of the data; another default (Regress reports White (HC1) standard errors unless asked otherwise, Baltagi reports classical ones); another small-sample correction (CR1 here, as in Stata; CR0 has none).
- **For your own method:** find a paper or textbook table computed on a public dataset, rerun the method on it, and print each number beside the published one in the same way.

## References

- Baltagi, B. H. (2005). *Econometric Analysis of Panel Data*, 3rd ed. Wiley.
- Munnell, A. H. (1990). Why Has Productivity Growth Declined? Productivity and Public Investment. *New England Economic Review*.
- Petersen, M. A. (2009). Estimating Standard Errors in Finance Panel Data Sets: Comparing Approaches. *Review of Financial Studies* 22(1), 435-480. [doi:10.1093/rfs/hhn053](https://doi.org/10.1093/rfs/hhn053)
- Petersen, M. A. [Test data with published standard errors](https://www.kellogg.northwestern.edu/faculty/petersen/htm/papers/se/test_data.htm).